In [1]:
from pathlib import Path
import subprocess

import pystac
import openeo

In [2]:
from openeo.rest.stac_resource import StacResource
from openeo.internal.graph_building import PGNode

### Connect to OpenEO backend

In [3]:
backend_url = "openeo-staging.dataspace.copernicus.eu"
#backend_url = "openeo.dataspace.copernicus.eu"
connection = openeo.connect(backend_url).authenticate_oidc()

Authenticated using refresh token.


## Query

In [4]:
collection_url = "https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l1c"

w,s,e,n = 10.6, 44.5, 11.0, 44.8
spatial_extent = { "west": w, "south": s, "east": e, "north": n}

query_pg = connection.datacube_from_process(
    "query_stac",
    url=collection_url,
    temporal_extent=["2026-04-06", "2026-04-09"],
    spatial_extent=spatial_extent
)

# optional, just to visualize
res = connection.execute(query_pg)
#import json
#Path("query_output.json").write_text(json.dumps(res))
res

{'features': [{'assets': {'B01': {'alternate': {'https': {'alternate:name': 'HTTPS',
       'auth:refs': ['oidc'],
       'href': 'https://download.dataspace.copernicus.eu/odata/v1/Products(d4ac876b-a4bc-429c-99ac-d4c11dabde1d)/Nodes(S2C_MSIL1C_20260407T101021_N0512_R022_T32TPQ_20260407T142059.SAFE)/Nodes(GRANULE)/Nodes(L1C_T32TPQ_A008286_20260407T101624)/Nodes(IMG_DATA)/Nodes(T32TPQ_20260407T101021_B01.jp2)/$value',
       'storage:refs': []}},
     'alternate:name': 'S3',
     'auth:refs': ['s3'],
     'bands': [{'description': 'Coastal aerosol (band 1)',
       'eo:center_wavelength': 0.443,
       'eo:common_name': 'coastal',
       'eo:full_width_half_max': 0.228,
       'name': 'B01'}],
     'data_type': 'uint16',
     'file:checksum': '1620add1f8b8fcbc122d9145002ebff56b68ab481bc28d37f2d694ba4670f3826301',
     'file:local_path': 'S2C_MSIL1C_20260407T101021_N0512_R022_T32TPQ_20260407T142059.SAFE/GRANULE/L1C_T32TPQ_A008286_20260407T101624/IMG_DATA/T32TPQ_20260407T101021_B01.jp2',


## Preparing CWL documents

> Note that this step will not be necessary when using a released version which will included the preprocessed files


In [5]:
cwl_root = Path("../cwl").resolve()
level2_res = subprocess.run(
    [
        "cwltool",
        "--pack",
        str(cwl_root / "force-l2-workflow.cwl"),
    ],
    capture_output=True
)
try:
    level2_res.check_returncode()
except subprocess.CalledProcessError as e:
    print("STDERR")
    print(level2_res.stderr.decode())
    raise e
cwl_level2 = level2_res.stdout.decode()

tsa_res = subprocess.run(
    [
        "cwltool",
        "--pack",
        str(cwl_root / "force-tsa-workflow.cwl"),
    ],
    capture_output=True
)
try:
    tsa_res.check_returncode()
except subprocess.CalledProcessError as e:
    print("STDERR")
    print(tsa_res.stderr.decode())
    raise e
cwl_tsa = tsa_res.stdout.decode()

## Running FORCE level 2 with `run_cwl_to_stac`

This is the workflow that should be used in the end. Catalogs are not yet accessible from the result (as shown below).
The process only completes successfully when ran on `openeo-staging` (2026-03-26)

> note that this part will be replaced by the force custom processes, so we run force_level2 instead of run_cwl_to_stac

In [6]:
aoi=f'{{ "type": "Feature", "geometry": {{ "type": "Polygon", "coordinates": [[[{w},{s}],[{w},{n}],[{e},{n}],[{e},{s}],[{w},{s}]]] }}, "properties": {{ "name": "Parma" }}, "id": "08" }}'
print(f"AOI:\n{aoi}")
context = dict(
    stac_document=query_pg,
    aoi=aoi,
)

stac_resource = StacResource(
    graph=PGNode(
        process_id="run_cwl_to_stac",
        arguments={
            "cwl_url": cwl_level2,
            "context": context,
        }
    ),
    connection=connection,
)
l2_job = stac_resource.create_job(title="FORCE level 2")
l2_job.start_and_wait()

AOI:
{ "type": "Feature", "geometry": { "type": "Polygon", "coordinates": [[[10.6,44.5],[10.6,44.8],[11.0,44.8],[11.0,44.5],[10.6,44.5]]] }, "properties": { "name": "Parma" }, "id": "08" }
0:00:00 Job 'j-2604201007334ba69cdc57288316e7c4': send 'start'
0:00:14 Job 'j-2604201007334ba69cdc57288316e7c4': created (progress 0%)
0:00:19 Job 'j-2604201007334ba69cdc57288316e7c4': created (progress 0%)
0:00:26 Job 'j-2604201007334ba69cdc57288316e7c4': created (progress 0%)
0:00:34 Job 'j-2604201007334ba69cdc57288316e7c4': created (progress 0%)
0:00:44 Job 'j-2604201007334ba69cdc57288316e7c4': running (progress 6.5%)
0:00:57 Job 'j-2604201007334ba69cdc57288316e7c4': running (progress 8.3%)
0:01:12 Job 'j-2604201007334ba69cdc57288316e7c4': running (progress 10.4%)
0:01:32 Job 'j-2604201007334ba69cdc57288316e7c4': running (progress 12.9%)
0:01:56 Job 'j-2604201007334ba69cdc57288316e7c4': running (progress 15.9%)
0:02:26 Job 'j-2604201007334ba69cdc57288316e7c4': running (progress 19.3%)
0:03:03 Job 

<BatchJob job_id='j-2604201007334ba69cdc57288316e7c4'>

In [7]:
l2_results = l2_job.get_results()
l2_results

<JobResults for job 'j-2604201007334ba69cdc57288316e7c4'>

## Download results

In [8]:
#l2_results.download_files("assets")

## Access catalog link

In [9]:
catalog_link = next(iter(l for l in l2_results.get_metadata()["links"] if l["href"].endswith("catalogue.json")))
catalog_url = f"https://{backend_url}/{catalog_link['href']}"
print("Assets:")
print("\t", l2_results.get_metadata()["assets"])
print("Catalog link:")
print("\t", catalog_link)

Assets:
	 {'europe/X0031_Y0029/20260407_LEVEL2_SEN2C_BOA.tif': {'eo:bands': [{'name': 'B2'}, {'name': 'B3'}, {'name': 'B4'}, {'name': 'B5'}, {'name': 'B6'}, {'name': 'B7'}, {'name': 'B8'}, {'name': 'B8A'}, {'name': 'B11'}, {'name': 'B12'}], 'file:size': 40038944, 'href': 'https://s3.stag.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-staging-waw4-1/batch_jobs/j-2604201007334ba69cdc57288316e7c4/europe/X0031_Y0029/20260407_LEVEL2_SEN2C_BOA.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=69a0f8afedce47fb8b28240b044dbe39%2F20260420%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260420T102327Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXN0YWdpbmctd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnN0YWcud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsia

The catalog indicated in the job results does not exist:

In [10]:
catalog_url

'https://openeo-staging.dataspace.copernicus.eu//batch_jobs/j-2604201007334ba69cdc57288316e7c4/catalogue.json'

In [11]:
!curl -I {catalog_url}

HTTP/1.1 404 Not Found
Content-Length: 153
Content-Type: text/html
Date: Mon, 20 Apr 2026 10:23:28 GMT



We can however access the catalog (and from it, the item) through the logs:

In [12]:
force_l2_logs = l2_job.logs()
filtered_logs = (l.message for l in force_l2_logs if "catalog.json" in l.message or "catalogue.json" in l.message)
print(f"\n{'-'*80}\n".join(filtered_logs))

                "location": "file:///calrissian/output-data/j-2604201007334ba69c-cal-cwl-ba6d3b0c/l2-ard/catalogue.json",
--------------------------------------------------------------------------------
                "path": "/calrissian/output-data/j-2604201007334ba69c-cal-cwl-ba6d3b0c/l2-ard/catalogue.json"
--------------------------------------------------------------------------------
                "basename": "catalogue.json",
--------------------------------------------------------------------------------
result 'l2-ard/catalogue.json': v.generate_public_url()='https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201007334ba69c-cal-cwl-ba6d3b0c/l2-ard/catalogue.json' v.generate_presigned_url()='https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201007334ba69c-cal-cwl-ba6d3b0c/l2-ard/catalogue.json?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=2b362f5724b14d019bbddaeee22f303e%2F20260420%2FWAW3-1%

In [13]:
log_with_item_url = next(iter(l.message for l in force_l2_logs if "copy_asset(" in l.message and ("catalogue.json" in l.message or "catalog.json" in l.message)))
log_with_item_url

'URL: copy_asset(https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201007334ba69c-cal-cwl-ba6d3b0c/l2-ard/catalogue.json)'

In [14]:
print(log_with_item_url)
catalog_url = log_with_item_url.removeprefix("URL: copy_asset(").removesuffix(")")
catalog_url

URL: copy_asset(https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201007334ba69c-cal-cwl-ba6d3b0c/l2-ard/catalogue.json)


'https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201007334ba69c-cal-cwl-ba6d3b0c/l2-ard/catalogue.json'

In [15]:
item = next(iter((pystac.Catalog.from_file(catalog_url).get_items())))
item_url = next(iter(item.get_links("self"))).href
item_url

'https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201007334ba69c-cal-cwl-ba6d3b0c/l2-ard/cube-20260420T1022-l2-ard.json'

In [16]:
item = pystac.read_file(item_url)
tiles = set([k.split(".")[1] for k in item.assets.keys() if k.endswith("BOA")])
tile_x = [int(tile[1:5]) for tile in tiles]
tile_y = [int(tile[7:]) for tile in tiles]
tile_x, tile_y

([31], [29])

### Running FORCE TSA

In [17]:
# process parameters
context = dict(
    stac_url=item_url,
    date_range=("2026-04-06", "2026-04-09"),
    x_tile_range=[min(tile_x), max(tile_x)],
    y_tile_range=[min(tile_y), max(tile_y)],
    stm=["AVG"],
    output_stm=True,
)

stac_resource = StacResource(
    graph=PGNode(
        process_id="run_cwl_to_stac",
        arguments={
            "cwl": cwl_tsa,
            "context": context,
        },
    ),
    connection=connection,
)
tsa_job = stac_resource.create_job(title="FORCE TSA")
tsa_job.start_and_wait()

0:00:00 Job 'j-2604201023314a218a3de25337c95262': send 'start'
0:00:14 Job 'j-2604201023314a218a3de25337c95262': created (progress 0%)
0:00:20 Job 'j-2604201023314a218a3de25337c95262': created (progress 0%)
0:00:26 Job 'j-2604201023314a218a3de25337c95262': created (progress 0%)
0:00:34 Job 'j-2604201023314a218a3de25337c95262': created (progress 0%)
0:00:44 Job 'j-2604201023314a218a3de25337c95262': running (progress 6.5%)
0:00:57 Job 'j-2604201023314a218a3de25337c95262': running (progress 8.3%)
0:01:12 Job 'j-2604201023314a218a3de25337c95262': running (progress 10.4%)
0:01:32 Job 'j-2604201023314a218a3de25337c95262': running (progress 12.9%)
0:01:56 Job 'j-2604201023314a218a3de25337c95262': running (progress 15.8%)
0:02:26 Job 'j-2604201023314a218a3de25337c95262': running (progress 19.3%)
0:03:03 Job 'j-2604201023314a218a3de25337c95262': running (progress 23.1%)
0:03:50 Job 'j-2604201023314a218a3de25337c95262': running (progress 27.5%)
0:04:49 Job 'j-2604201023314a218a3de25337c95262': r

<BatchJob job_id='j-2604201023314a218a3de25337c95262'>

In [18]:
tsa_results = tsa_job.get_results()
tsa_results

<JobResults for job 'j-2604201023314a218a3de25337c95262'>

In [19]:
tsa_logs = tsa_job.logs()
filtered_logs = (l.message for l in tsa_logs if "catalog.json" in l.message or "catalogue.json" in l.message)
print(f"\n{'-'*80}\n".join(filtered_logs))

     2464      1 -rw-r--r--   1 ubuntu   ubuntu        335 Apr 20 10:30 outputs/force-tsa/catalog.json
--------------------------------------------------------------------------------
                "location": "file:///calrissian/output-data/j-2604201023314a218a-cal-cwl-8172b2ef/force-tsa/catalog.json",
--------------------------------------------------------------------------------
                "basename": "catalog.json",
--------------------------------------------------------------------------------
                "path": "/calrissian/output-data/j-2604201023314a218a-cal-cwl-8172b2ef/force-tsa/catalog.json"
--------------------------------------------------------------------------------
result 'force-tsa/catalog.json': v.generate_public_url()='https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201023314a218a-cal-cwl-8172b2ef/force-tsa/catalog.json' v.generate_presigned_url()='https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-

In [20]:
log_with_catalog_url = next(iter(l.message for l in tsa_logs if "copy_asset(" in l.message and ("catalogue.json" in l.message or "catalog.json" in l.message)))
print(log_with_catalog_url)
catalog_url = log_with_catalog_url.removeprefix("URL: copy_asset(").removesuffix(")")
catalog_url

URL: copy_asset(https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201023314a218a-cal-cwl-8172b2ef/force-tsa/catalog.json)


'https://s3.waw4-1.cloudferro.com/calrissian/pvc-91dd9cc8-952b-4f4a-abf9-77a9e32beb58/j-2604201023314a218a-cal-cwl-8172b2ef/force-tsa/catalog.json'